# PAE Heatmaps — RAB11A × EXOC6 (rab11a_exoc6_0)

Reads every sample in `json_interactors/rab11a_exoc6_0/` and plots the
Predicted Aligned Error (PAE) matrix for each.

| Chain | Identity | Residues |
|-------|----------|----------|
| A | small molecule / ligand | 27 tokens |
| B | small molecule / ligand | 27 tokens |
| F | EXOC6 | 804 |
| I | RAB11A | 216 |

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
DATA_DIR = Path(r"N:\08_NK_structure_prediction\data\Exocyst_complex\json_interactors\rab11a_exoc6_0")

# Chain → gene name mapping for this prediction
CHAIN_GENE = {
    "A": "ligand-A",
    "B": "ligand-B",
    "F": "EXOC6",
    "I": "RAB11A",
}

# Chains excluded from PAE heatmaps
LIGAND_CHAINS = {"A", "B"}

FIGURE_DIR = (
    Path(r"N:\08_NK_structure_prediction\XL_MOPLC\XL_complex_structure\result")
    / "pae_rab11a_exoc6"
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figure output → {FIGURE_DIR}")

In [ ]:
# ── Nature figure style ───────────────────────────────────────────────────────
mpl.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":          7,
    "axes.titlesize":     8,
    "axes.labelsize":     7,
    "xtick.labelsize":    6,
    "ytick.labelsize":    6,
    "legend.fontsize":    6,
    "legend.frameon":     False,
    "axes.linewidth":     0.5,
    "xtick.major.width":  0.5,
    "ytick.major.width":  0.5,
    "xtick.major.size":   2.5,
    "ytick.major.size":   2.5,
    "xtick.direction":    "out",
    "ytick.direction":    "out",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "lines.linewidth":    0.75,
    "patch.linewidth":    0.5,
    "figure.dpi":         150,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.05,
    "pdf.fonttype":       42,
    "ps.fonttype":        42,
})

NC1 = 89  / 25.4
NC2 = 183 / 25.4


def save_fig(fig, name):
    path = FIGURE_DIR / f"{name}.svg"
    fig.savefig(path, format="svg")
    print(f"Saved → {path}")

In [ ]:
# ── Helper: load one confidences.json ────────────────────────────────────────
def load_confidences(path):
    """Return (token_chain_ids, token_res_ids, pae_array, atom_plddts)."""
    with open(path) as fh:
        d = json.load(fh)
    return (
        np.array(d["token_chain_ids"]),
        np.array(d["token_res_ids"], dtype=int),
        np.array(d["pae"]),
        np.array(d.get("atom_plddts", [])),
    )


def load_summary(path):
    with open(path) as fh:
        return json.load(fh)


def chain_boundaries(token_chains):
    """Return boundary indices where chain changes."""
    bounds = [0]
    for i in range(1, len(token_chains)):
        if token_chains[i] != token_chains[i - 1]:
            bounds.append(i)
    bounds.append(len(token_chains))
    return bounds


def compute_iptm(pae, token_chains, cutoff=12.0):
    n  = len(token_chains)
    d0 = max(1.24 * max(n - 15, 1) ** (1 / 3) - 1.8, 0.1)
    inter  = token_chains[:, None] != token_chains[None, :]
    intf   = inter & ((pae < cutoff) | (pae.T < cutoff))
    n_intf = int(intf.sum())
    if n_intf == 0:
        return np.nan
    tm = 1.0 / (1.0 + (pae / d0) ** 2)
    return float((tm * intf).sum() / n_intf)


def filter_ligand(tc, pae, exclude=None):
    """Remove ligand tokens from tc and slice the corresponding PAE rows/cols."""
    if exclude is None:
        exclude = LIGAND_CHAINS
    mask = np.array([c not in exclude for c in tc])
    return tc[mask], pae[np.ix_(mask, mask)]

In [ ]:
# ── Collect all samples ───────────────────────────────────────────────────────
samples = []

# Load ranking scores
ranking_csv = DATA_DIR / "ranking_scores.csv"
if ranking_csv.exists():
    ranking_df = pd.read_csv(ranking_csv)
    ranking_df["label"] = ranking_df.apply(
        lambda r: f"seed-{int(r['seed'])}_sample-{int(r['sample'])}", axis=1
    )
    rank_map = dict(zip(ranking_df["label"], ranking_df["ranking_score"]))
    print(ranking_df.to_string(index=False))
else:
    rank_map = {}

# Scan subdirectories for samples
for subdir in sorted(DATA_DIR.iterdir()):
    if not subdir.is_dir():
        continue
    conf = subdir / "confidences.json"
    summ = subdir / "summary_confidences.json"
    if not conf.exists():
        continue
    label = subdir.name
    tc, tr, pae, plddts = load_confidences(conf)
    s = load_summary(summ) if summ.exists() else {}
    samples.append(dict(
        label       = label,
        tc          = tc,
        tr          = tr,
        pae         = pae,
        plddts      = plddts,
        summary     = s,
        rank_score  = rank_map.get(label, np.nan),
    ))

print(f"\nLoaded {len(samples)} samples")
for s in samples:
    print(f"  {s['label']:30s}  PAE={s['pae'].shape}  rank={s['rank_score']:.4f}  "
          f"ipTM(summary)={s['summary'].get('iptm', 'n/a')}")

In [ ]:
# ── Figure: PAE heatmaps for all samples ─────────────────────────────────────
n_samples = len(samples)
ncols     = min(3, n_samples)
nrows     = (n_samples + ncols - 1) // ncols
panel_w   = NC2 / ncols

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(NC2, panel_w * nrows),
    squeeze=False,
    constrained_layout=True,
)

panel_labels = "abcdefghijklmnopqrstuvwxyz"

for idx, sample in enumerate(samples):
    row, col = divmod(idx, ncols)
    ax  = axes[row][col]
    pae = sample["pae"]
    tc  = sample["tc"]
    tc, pae = filter_ligand(tc, pae)
    N   = len(tc)

    im = ax.imshow(
        pae, cmap="RdYlGn_r", vmin=0, vmax=30,
        aspect="equal", interpolation="nearest", rasterized=True,
    )

    # Chain boundaries
    bounds = chain_boundaries(tc)
    for b in bounds[1:-1]:
        ax.axhline(b - 0.5, color="k", linewidth=0.5, alpha=0.8)
        ax.axvline(b - 0.5, color="k", linewidth=0.5, alpha=0.8)

    # Gene-name axis labels
    unique_chains = list(dict.fromkeys(tc))
    offset = N * 0.05
    for k, c in enumerate(unique_chains):
        mid  = (bounds[k] + bounds[k + 1]) / 2
        name = CHAIN_GENE.get(c, c)
        ax.text(mid, -offset, name,
                ha="center", va="center", fontsize=5.5,
                rotation=30, clip_on=False)
        ax.text(-offset, mid, name,
                ha="center", va="center", fontsize=5.5,
                rotation=90, clip_on=False)

    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(0.4)

    iptm_val  = sample["summary"].get("iptm", np.nan)
    rank_val  = sample["rank_score"]
    iptm_str  = f"{iptm_val:.3f}" if not (isinstance(iptm_val, float) and np.isnan(iptm_val)) else "n/a"
    rank_str  = f"{rank_val:.4f}" if not np.isnan(rank_val) else "n/a"

    ax.set_xlabel(
        f"{sample['label']}\nipTM={iptm_str}  rank={rank_str}",
        labelpad=6,
    )

    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02, shrink=0.85)
    cb.set_label("PAE (Å)", labelpad=2)
    cb.ax.tick_params(labelsize=5, width=0.4, length=2)
    cb.outline.set_linewidth(0.4)

    ax.text(-0.08, 1.04, panel_labels[idx], transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top")

for idx in range(n_samples, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.suptitle(
    "Predicted Aligned Error (PAE) — RAB11A × EXOC6  (all samples)",
    fontsize=8,
)
save_fig(fig, "pae_heatmaps_all_samples")
plt.show()

**Fig. 1 | Predicted aligned error (PAE) heatmaps for all AlphaFold predictions of the RAB11A–EXOC6 complex.**
Each panel shows the PAE matrix (Å) for one independent AlphaFold prediction (identified by seed–sample combination). Ligand chains (A and B) are excluded; only EXOC6 (chain F, 804 residues) and RAB11A (chain I, 216 residues) are displayed. Rows and columns represent residues in chain order; the solid black line marks the inter-protein chain boundary. The colour scale runs from 0 (green, high confidence in relative position) to 30 Å (red). The panel subtitle reports the sample identifier, ipTM score (from `summary_confidences.json`), and AlphaFold ranking score. Panels are labelled **a–z** in order of iteration.


In [ ]:
# ── Figure: best-ranked sample (large, single panel) ─────────────────────────
best = max(samples, key=lambda s: s["rank_score"] if not np.isnan(s["rank_score"]) else -1)
print(f"Best sample: {best['label']}  rank={best['rank_score']:.4f}")

pae = best["pae"]
tc  = best["tc"]
tc, pae = filter_ligand(tc, pae)
N   = len(tc)
bounds = chain_boundaries(tc)

fig, ax = plt.subplots(figsize=(NC1 * 1.3, NC1 * 1.3))

im = ax.imshow(
    pae, cmap="RdYlGn_r", vmin=0, vmax=30,
    aspect="equal", interpolation="nearest", rasterized=True,
)

# Chain boundaries
for b in bounds[1:-1]:
    ax.axhline(b - 0.5, color="k", linewidth=0.6, alpha=0.9)
    ax.axvline(b - 0.5, color="k", linewidth=0.6, alpha=0.9)

# Shaded inter-chain region highlight (F–I interface)
unique_chains = list(dict.fromkeys(tc))
offset = N * 0.04
for k, c in enumerate(unique_chains):
    mid  = (bounds[k] + bounds[k + 1]) / 2
    name = CHAIN_GENE.get(c, c)
    ax.text(mid, -offset, name,
            ha="center", va="center", fontsize=7,
            rotation=30, clip_on=False, fontweight="bold")
    ax.text(-offset, mid, name,
            ha="center", va="center", fontsize=7,
            rotation=90, clip_on=False, fontweight="bold")

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_linewidth(0.5)

iptm_val = best["summary"].get("iptm", np.nan)
iptm_str = f"{iptm_val:.3f}" if not (isinstance(iptm_val, float) and np.isnan(iptm_val)) else "n/a"
ax.set_title(
    f"RAB11A × EXOC6  |  {best['label']}  |  ipTM = {iptm_str}  |  ranking score = {best['rank_score']:.4f}",
    pad=8, fontsize=7,
)

cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
cb.set_label("PAE (Å)", fontsize=7)
cb.ax.tick_params(labelsize=6, width=0.5, length=2.5)
cb.outline.set_linewidth(0.5)

ax.text(-0.06, 1.04, "a", transform=ax.transAxes,
        fontsize=10, fontweight="bold", va="top")

save_fig(fig, "pae_heatmap_best_sample")
plt.show()

**Fig. 2 | PAE heatmap of the highest-ranked AlphaFold model for the RAB11A–EXOC6 complex.**
The PAE matrix is displayed for the sample with the highest AlphaFold ranking score among all predicted seeds and samples. Ligand chains are excluded; axes represent residues of EXOC6 (chain F) and RAB11A (chain I), labelled in bold. The solid line marks the chain boundary; the colour scale ranges from 0 to 30 Å (green–red). Low off-diagonal PAE values in the inter-chain quadrants indicate high confidence in the predicted relative orientation of the two proteins. The figure title reports the sample identifier, ipTM score, and ranking score.


In [ ]:
# ── Figure: average PAE across all samples ───────────────────────────────────
pae_stack = np.stack([filter_ligand(s["tc"], s["pae"])[1] for s in samples], axis=0)
pae_mean  = pae_stack.mean(axis=0)
pae_std   = pae_stack.std(axis=0)

tc, _ = filter_ligand(samples[0]["tc"], samples[0]["pae"])
N  = len(tc)
bounds = chain_boundaries(tc)
unique_chains = list(dict.fromkeys(tc))

fig, axes = plt.subplots(1, 2, figsize=(NC2 * 0.75, NC2 * 0.35),
                         constrained_layout=True)

for ax, data, title, panel in zip(
    axes,
    [pae_mean, pae_std],
    ["Mean PAE (Å)",  "Std PAE (Å)"],
    ["a", "b"],
):
    vmax = 30 if "Mean" in title else pae_std.max()
    cmap = "RdYlGn_r" if "Mean" in title else "YlOrRd"
    im = ax.imshow(
        data, cmap=cmap, vmin=0, vmax=vmax,
        aspect="equal", interpolation="nearest", rasterized=True,
    )
    for b in bounds[1:-1]:
        ax.axhline(b - 0.5, color="k", linewidth=0.5, alpha=0.8)
        ax.axvline(b - 0.5, color="k", linewidth=0.5, alpha=0.8)

    offset = N * 0.05
    for k, c in enumerate(unique_chains):
        mid  = (bounds[k] + bounds[k + 1]) / 2
        name = CHAIN_GENE.get(c, c)
        ax.text(mid, -offset, name, ha="center", va="center",
                fontsize=5.5, rotation=30, clip_on=False)
        ax.text(-offset, mid, name, ha="center", va="center",
                fontsize=5.5, rotation=90, clip_on=False)

    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(0.4)

    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02, shrink=0.85)
    cb.set_label(title, labelpad=2)
    cb.ax.tick_params(labelsize=5, width=0.4, length=2)
    cb.outline.set_linewidth(0.4)

    ax.text(-0.08, 1.04, panel, transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top")

fig.suptitle(
    f"Mean & Std PAE across {len(samples)} samples — RAB11A × EXOC6",
    fontsize=8,
)
save_fig(fig, "pae_mean_std")
plt.show()

**Fig. 3 | Ensemble mean and standard deviation of predicted aligned error for the RAB11A–EXOC6 complex.**
**(a)** Mean PAE matrix (Å) averaged across all predicted samples. Low values (green) in the off-diagonal quadrants (EXOC6–RAB11A interface regions) indicate consistently high inter-chain positional confidence. **(b)** Standard deviation of the PAE matrix across all samples (YlOrRd scale); low values indicate reproducibility of the predicted relative positions across independent seeds. Chain boundaries are marked by solid lines; axes are labelled by protein name. Ligand chains are excluded from both panels. *N* denotes the total number of samples used for averaging.


In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for s in samples:
    summ = s["summary"]
    rows.append({
        "sample"       : s["label"],
        "ranking_score": round(s["rank_score"], 4) if not np.isnan(s["rank_score"]) else np.nan,
        "iptm"         : summ.get("iptm", np.nan),
        "ptm"          : summ.get("ptm", np.nan),
        "frac_disorder": summ.get("fraction_disordered", np.nan),
        "has_clash"    : summ.get("has_clash", np.nan),
    })

summary_df = (
    pd.DataFrame(rows)
    .sort_values("ranking_score", ascending=False)
    .reset_index(drop=True)
)
print(summary_df.to_string(index=False))